In [2]:
"""
Quick Duplicate Detection — Module 3 
"""

import glob
import pandas as pd
import numpy as np
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data')
EXPORTS_DIR  = BASE_DIR / 'Exports'

KEY_COLS = ["Period", "Commodity", "Province", "Country", "State"]

# ──────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────
def load_raw(filepath):
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()
    
    # Remove footer lines (metadata at bottom of file)
    if "Period" in df.columns:
        # Step 1: Remove rows where Period contains metadata keywords
        metadata_keywords = ["source:", "note:", "notes:", "data source:", "disclaimer:", 
                            "citation:", "data extracted:", "extracted:", "date:", 
                            "© ", "copyright", "confidential"]
        period_lower = df["Period"].fillna("").astype(str).str.lower().str.strip()
        mask = period_lower.str.startswith(tuple(metadata_keywords))
        df = df[~mask]
        
        # Step 2: Remove rows where Period is not a valid date format
        period_values = df["Period"].fillna("").astype(str).str.strip()
        valid_mask = (
            (period_values == "") |  # Empty is ok (will be caught as missing value)
            (period_values.str.match(r'^\d{4}-\d{2}-\d{2}')) |  # YYYY-MM-DD
            (period_values.str.match(r'^\d{4}/\d{2}/\d{2}')) |  # YYYY/MM/DD
            (period_values.str.match(r'^\d{2}-\d{2}-\d{4}')) |  # DD-MM-YYYY
            (pd.to_datetime(df["Period"], errors='coerce').notna())  # Any parseable date
        )
        df = df[valid_mask]
        
        # Step 3: Remove trailing completely empty rows
        last_valid_idx = None
        for idx in reversed(df.index):
            row = df.loc[idx]
            has_data = row.notna().any() and (row.astype(str).str.strip() != "").any()
            if has_data:
                last_valid_idx = idx
                break
        
        if last_valid_idx is not None:
            df = df.loc[:last_valid_idx]
    
    df["_source_file"] = Path(filepath).name
    return df

def load_all(files):
    frames = []
    errors = []
    for f in files:
        try:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors

# ──────────────────────────────────────────────────────────────
# MODULE 3 — DUPLICATE DETECTION
# ──────────────────────────────────────────────────────────────
def module_duplicates(raw):
    df = raw.drop(columns=["_source_file"], errors="ignore")
    n = len(df)
    
    results = {
        "summary": [],
        "examples": {}
    }
    
    # ═══════════════════════════════════════════════════════════
    # 1. FULL ROW DUPLICATES (exact same data in every column)
    # ═══════════════════════════════════════════════════════════
    full_dup_mask = df.duplicated(keep=False)  # Mark ALL duplicates (not just subsequent)
    full_dup_count = df.duplicated().sum()  # Count only subsequent (not first occurrence)
    full_dup_pct = (full_dup_count / n * 100).round(2) if n > 0 else 0
    
    # Determine severity based on count
    if full_dup_count == 0:
        severity = "✅ OK"
        action = "None"
    elif full_dup_count < 10:
        severity = "🟡 WARN"
        action = "Review and remove duplicates"
    else:
        severity = "🔴 CRITICAL"
        action = "Investigate source — likely data export issue"
    
    results["summary"].append({
        "check": "Full row duplicates",
        "description": "Exact same data in every column",
        "count": int(full_dup_count),
        "pct_of_total": full_dup_pct,
        "severity": severity,
        "action": action
    })
    
    # Store example full duplicates (show the duplicated rows, not the originals)
    if full_dup_count > 0:
        # Get rows that are duplicates (not the first occurrence)
        dup_rows = raw[df.duplicated(keep='first')].head(5)
        display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                       if c in dup_rows.columns]
        results["examples"]["full_duplicates"] = dup_rows[display_cols].copy()
    
    # ═══════════════════════════════════════════════════════════
    # 2. KEY-BASED DUPLICATES (same Period+Commodity+Province+Country)
    # ═══════════════════════════════════════════════════════════
    available_key_cols = [c for c in KEY_COLS if c in df.columns]
    
    if available_key_cols:
        key_dup_mask = df.duplicated(subset=available_key_cols, keep=False)
        key_dup_count = df.duplicated(subset=available_key_cols).sum()
        key_dup_pct = (key_dup_count / n * 100).round(2) if n > 0 else 0
        
        # Severity based on percentage
        if key_dup_count == 0:
            severity = "✅ OK"
            action = "None"
        elif key_dup_pct < 5:
            severity = "🟡 WARN"
            action = "Review — may be legitimate (multiple shipments same day)"
        elif key_dup_pct < 15:
            severity = "🔴 HIGH"
            action = "Investigate — too many duplicates for same commodity/country/date"
        else:
            severity = "🔴 CRITICAL"
            action = "Critical issue — likely systematic data duplication"
        
        results["summary"].append({
            "check": "Key-based duplicates",
            "description": f"Same {', '.join(available_key_cols)} but different Value/Qty",
            "count": int(key_dup_count),
            "pct_of_total": key_dup_pct,
            "severity": severity,
            "action": action
        })
        
        # Store example key duplicates with grouping details
        if key_dup_count > 0:
            # Show the groups with duplicates
            dup_groups = (
                df[key_dup_mask]
                .groupby(available_key_cols, dropna=False)
                .size()
                .reset_index(name='occurrence_count')
                .sort_values('occurrence_count', ascending=False)
                .head(5)
            )
            results["examples"]["key_duplicate_groups"] = dup_groups
            
            # Show actual duplicate rows (first group only, for detail)
            if len(dup_groups) > 0:
                first_group = dup_groups.iloc[0][available_key_cols].to_dict()
                mask = pd.Series([True] * len(df))
                for col, val in first_group.items():
                    if pd.isna(val):
                        mask &= df[col].isna()
                    else:
                        mask &= (df[col] == val)
                
                dup_rows_detail = raw[mask].head(5)
                display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                               if c in dup_rows_detail.columns]
                results["examples"]["key_duplicates_detail"] = dup_rows_detail[display_cols].copy()
    
    # ═══════════════════════════════════════════════════════════
    # 3. VALUE+QTY DUPLICATES (same keys + same Value + same Qty)
    # ═══════════════════════════════════════════════════════════
    vq_cols = [c for c in available_key_cols + ["Value ($)", "Quantity"] if c in df.columns]
    
    if len(vq_cols) >= 4:  # Need at least some key cols + value/qty
        vq_dup_mask = df.duplicated(subset=vq_cols, keep=False)
        vq_dup_count = df.duplicated(subset=vq_cols).sum()
        vq_dup_pct = (vq_dup_count / n * 100).round(2) if n > 0 else 0
        
        if vq_dup_count == 0:
            severity = "✅ OK"
            action = "None"
        elif vq_dup_count < 5:
            severity = "🟡 WARN"
            action = "Review — likely data entry duplicates"
        else:
            severity = "🔴 HIGH"
            action = "Remove — identical records are almost certainly duplicates"
        
        results["summary"].append({
            "check": "Value+Qty duplicates",
            "description": "Same key fields + same Value ($) + same Quantity",
            "count": int(vq_dup_count),
            "pct_of_total": vq_dup_pct,
            "severity": severity,
            "action": action
        })
        
        # Store examples
        if vq_dup_count > 0:
            dup_rows = raw[df.duplicated(subset=vq_cols, keep='first')].head(5)
            display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                           if c in dup_rows.columns]
            results["examples"]["value_qty_duplicates"] = dup_rows[display_cols].copy()
    
    return results

# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(results):
    print("\n" + "="*80)
    print("  MODULE 3 — DUPLICATE DETECTION")
    print("="*80)
    
    # ─────────────────────────────────────────────────────────
    # SUMMARY TABLE
    # ─────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("  DUPLICATE SUMMARY")
    print("─"*80 + "\n")
    
    for i, r in enumerate(results["summary"], 1):
        print(f"{i}. {r['check']}")
        print(f"   Description: {r['description']}")
        print(f"   Count:       {r['count']:,} duplicates ({r['pct_of_total']}% of total)")
        print(f"   Severity:    {r['severity']}")
        print(f"   Action:      {r['action']}")
        print()
    
    # ─────────────────────────────────────────────────────────
    # DETAILED EXAMPLES
    # ─────────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("  DETAILED EXAMPLES")
    print("="*80)
    
    # Full duplicates
    if "full_duplicates" in results["examples"]:
        print("\n" + "─"*80)
        print("  1. FULL ROW DUPLICATES — Example Rows (showing up to 5)")
        print("─"*80 + "\n")
        
        df_example = results["examples"]["full_duplicates"]
        for idx, row in df_example.iterrows():
            print(f"   Row {idx} (duplicate):")
            for col in df_example.columns:
                val = row[col]
                if pd.isna(val):
                    val_str = "[EMPTY]"
                elif str(val).strip() == "":
                    val_str = "[EMPTY]"
                else:
                    val_str = str(val)[:50]
                print(f"      {col:20s}: {val_str}")
            print()
    else:
        print("\n✅ No full row duplicates found\n")
    
    # Key-based duplicates - show groups
    if "key_duplicate_groups" in results["examples"]:
        print("\n" + "─"*80)
        print("  2. KEY-BASED DUPLICATES — Top Groups (showing up to 5)")
        print("─"*80 + "\n")
        print("   These commodity-country-date combinations appear multiple times:\n")
        
        df_groups = results["examples"]["key_duplicate_groups"]
        for idx, row in df_groups.iterrows():
            print(f"   Group {idx + 1}:")
            for col in df_groups.columns:
                if col != "occurrence_count":
                    val = row[col]
                    val_str = str(val) if pd.notna(val) else "[EMPTY]"
                    print(f"      {col:20s}: {val_str}")
            print(f"      {'Occurrences':20s}: {int(row['occurrence_count'])} times")
            print()
        
        # Show detail for first group
        if "key_duplicates_detail" in results["examples"]:
            print("\n   📋 Detail for first group (showing all occurrences):\n")
            df_detail = results["examples"]["key_duplicates_detail"]
            
            for idx, row in df_detail.iterrows():
                print(f"   Row {idx}:")
                for col in df_detail.columns:
                    val = row[col]
                    val_str = str(val)[:50] if pd.notna(val) else "[EMPTY]"
                    print(f"      {col:20s}: {val_str}")
                print()
    else:
        print("\n✅ No key-based duplicates found\n")
    

# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*80)
    print("  DUPLICATE DETECTION — MODULE 3 ONLY")
    print("="*80)

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)
    print(f"\nFound {len(export_files)} CSV file(s) in {EXPORTS_DIR}")

    if not export_files:
        print("❌ No files found. Check EXPORTS_DIR path.")
        exit(1)

    print("Loading files...")
    raw, load_errors = load_all(export_files)
    print(f"Loaded {len(raw):,} rows from {len(set(raw['_source_file']))} file(s)")
    if load_errors:
        print(f"⚠️  {len(load_errors)} file(s) failed to load")

    print("\nRunning duplicate detection...")
    results = module_duplicates(raw)
    
    print_results(results)
    
    # Summary
    total_issues = sum(r["count"] for r in results["summary"])
    critical_checks = sum(1 for r in results["summary"] if "🔴" in r["severity"])
    
    print("="*80)
    print(f"  SUMMARY:")
    print(f"    Total duplicate records: {total_issues:,}")
    print(f"    Critical/High severity checks: {critical_checks}/{len(results['summary'])}")
    print("="*80 + "\n")


  DUPLICATE DETECTION — MODULE 3 ONLY

Found 154 CSV file(s) in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
Loading files...
Loaded 2,536,465 rows from 154 file(s)

Running duplicate detection...

  MODULE 3 — DUPLICATE DETECTION

────────────────────────────────────────────────────────────────────────────────
  DUPLICATE SUMMARY
────────────────────────────────────────────────────────────────────────────────

1. Full row duplicates
   Description: Exact same data in every column
   Count:       0 duplicates (0.0% of total)
   Severity:    ✅ OK
   Action:      None

2. Key-based duplicates
   Description: Same Period, Commodity, Province, Country, State but different Value/Qty
   Count:       0 duplicates (0.0% of total)
   Severity:    ✅ OK
   Action:      None

3. Value+Qty duplicates
   Description: Same key fields + same Value ($) + same Quantity
   Count:       0 duplicates (0.0% of total)
   Severity:    ✅ OK
   Action:      None


  DETAILED EXAMPLES

✅ No 